# 3D Neural FEM Surrogate — MeshGraphNet (Kaggle)

Trains the MeshGraphNet on the cantilever-beam FEA dataset on a Kaggle GPU.

## One-time setup
1. **Settings → Accelerator → GPU P100** (recommended — beats T4×2 for this scatter-heavy GNN).
2. **Settings → Internet → On** (needed for the pip install in the first cell).
3. **Add data** (right sidebar) → your Kaggle dataset that contains `graph_cache_k12_v2.pt`.
4. Edit `CACHE_PATH` in cell 3 to match your dataset's slug.


In [ ]:
!pip install -q torch_geometric


In [ ]:
# Reduce CUDA allocator fragmentation - needed on T4 (16 GB) with this GNN.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  memory: {props.total_memory/1e9:.1f} GB")


In [ ]:
# Path to the uploaded graph cache. Edit the dataset slug to match yours.
CACHE_PATH = Path("/kaggle/input/fem-surrogate-graphs/graph_cache_k12_v2.pt")

graphs, sample_ids = torch.load(CACHE_PATH, weights_only=False)
print(f"{len(graphs)} graphs loaded")
g = graphs[0]
print(f"  example: x={tuple(g.x.shape)}  edges={tuple(g.edge_index.shape)}  y={tuple(g.y.shape)}")


In [ ]:
# Deterministic 70/15/15 split
random.seed(42)
idx = list(range(len(graphs)))
random.shuffle(idx)
n = len(graphs); n_tr = int(0.7 * n); n_va = int(0.15 * n)
splits = {
    "train": [graphs[i] for i in idx[:n_tr]],
    "val":   [graphs[i] for i in idx[n_tr:n_tr + n_va]],
    "test":  [graphs[i] for i in idx[n_tr + n_va:]],
}
print({k: len(v) for k, v in splits.items()})

# Normalisation stats computed on training targets only
y_train = torch.cat([g.y for g in splits["train"]], dim=0)
y_mean = y_train.mean(0)
y_std  = y_train.std(0)
print("y_mean:", y_mean.numpy(), "y_std:", y_std.numpy())


In [ ]:
# Normalise y in place on the CPU. We do NOT pre-move graphs to the GPU on
# CUDA: thousands of tiny per-graph allocations fragment the CUDA allocator
# and inflate ~1 GB of data into 13+ GB of reserved memory, OOMing before
# training even starts. (The pre-move trick only helps on Apple Silicon's
# unified memory.) DataLoader's pin_memory + non_blocking transfer handles
# the per-batch copy efficiently for us.
for sp, gs in splits.items():
    for g in gs:
        g.y = (g.y - y_mean) / y_std
print("graphs normalised (on CPU); will be moved to", device, "per batch")


In [ ]:
# --- MeshGraphNet (inlined from ml/gnn_model.py) -------------------------
def _mlp(in_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, out_dim), nn.GELU(),
        nn.Linear(out_dim, out_dim), nn.LayerNorm(out_dim),
    )

class _GNNLayer(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.edge_mlp = _mlp(3 * latent_dim, latent_dim)
        self.node_mlp = _mlp(2 * latent_dim, latent_dim)
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        new_edge = self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], dim=-1))
        edge_attr = edge_attr + new_edge
        agg = x.new_zeros(x.size(0), edge_attr.size(1))
        agg.scatter_add_(0, dst.unsqueeze(1).expand_as(edge_attr), edge_attr)
        x = x + self.node_mlp(torch.cat([x, agg], dim=-1))
        return x, edge_attr

class MeshGraphNet(nn.Module):
    def __init__(self, in_node_dim=12, in_edge_dim=4, out_dim=3,
                 latent_dim=128, n_mp_steps=10):
        super().__init__()
        self.node_encoder = _mlp(in_node_dim, latent_dim)
        self.edge_encoder = _mlp(in_edge_dim, latent_dim)
        self.layers  = nn.ModuleList([_GNNLayer(latent_dim) for _ in range(n_mp_steps)])
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim), nn.GELU(),
            nn.Linear(latent_dim, out_dim),
        )
    def forward(self, data):
        x = self.node_encoder(data.x)
        e = self.edge_encoder(data.edge_attr)
        for layer in self.layers:
            x, e = layer(x, data.edge_index, e)
        return self.decoder(x)

# n_mp_steps=12 (was 10): two extra message-passing steps give the model
# more reach across the mesh. Helps the high-variance stress field (R^2 0.84
# in the first run) more than the already-strong displacement fields.
# Will OOM on T4 16 GB past ~14; 12 is the safe headroom with fp16 batch=16.
model = MeshGraphNet(n_mp_steps=12).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")


In [ ]:
# Hyper-parameters
BATCH_SIZE       = 16         # T4 16 GB OOMs at 32 (10 MP layers x bf16 activations); 16 is the sweet spot
EVAL_BATCH_SIZE  = 64
EPOCHS           = 300
PATIENCE         = 50
LR               = 1e-3
AMP_DTYPE        = torch.float16    # T4 has fp16 Tensor Cores (no native bf16); needs GradScaler

loaders = {
    "train": DataLoader(splits["train"], batch_size=BATCH_SIZE,       shuffle=True,
                        num_workers=2, persistent_workers=True, pin_memory=True),
    "val":   DataLoader(splits["val"],   batch_size=EVAL_BATCH_SIZE,
                        num_workers=2, persistent_workers=True, pin_memory=True),
    "test":  DataLoader(splits["test"],  batch_size=EVAL_BATCH_SIZE,
                        num_workers=2, persistent_workers=True, pin_memory=True),
}
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# fp16 underflows easily on small gradients; GradScaler keeps them representable.
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
# Per-channel weighted Huber loss. Channels are [ux, uz, von_mises];
# 2x weight on von_mises pushes the optimizer to spend more capacity on the
# stress field, trading a small displacement-R^2 hit for better stress MARE.
CHANNEL_WEIGHTS = torch.tensor([1.0, 1.0, 2.0])

class _WeightedHuberLoss(nn.Module):
    def __init__(self, weights, delta=1.0):
        super().__init__()
        self.register_buffer("weights", weights.float())
        self.delta = delta
    def forward(self, pred, target):
        err = pred - target
        abs_err = err.abs()
        quad = 0.5 * err * err
        lin  = self.delta * (abs_err - 0.5 * self.delta)
        per_elem = torch.where(abs_err <= self.delta, quad, lin)
        return (per_elem * self.weights.to(per_elem.device)).mean()

criterion = _WeightedHuberLoss(CHANNEL_WEIGHTS, delta=1.0)
mse_metric = nn.MSELoss()

OUT_DIR  = Path("/kaggle/working")
BEST_PT  = OUT_DIR / "best_gnn.pt"

@torch.no_grad()
def evaluate(loader, crit):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE,
                            enabled=device.type == "cuda"):
            tot += crit(model(batch), batch.y).item()
        n += 1
    return tot / n


In [ ]:
best_val, best_epoch, stale = float("inf"), 0, 0

outer = tqdm(range(1, EPOCHS + 1), desc="training", unit="epoch", position=0)
for epoch in outer:
    model.train()
    loss_sum, n_steps = 0.0, 0
    inner = tqdm(
        loaders["train"],
        desc=f"  epoch {epoch:3d}/{EPOCHS}",
        unit="batch",
        leave=False,
        position=1,
    )
    for step, batch in enumerate(inner, 1):
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE,
                            enabled=device.type == "cuda"):
            loss = criterion(model(batch), batch.y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item(); n_steps += 1
        inner.set_postfix(
            step=f"{step}/{len(loaders['train'])}",
            loss=f"{loss.item():.4f}",
            avg=f"{loss_sum / n_steps:.4f}",
        )
    inner.close()
    train_loss = loss_sum / max(n_steps, 1)

    val_loss = evaluate(loaders["val"], criterion)
    scheduler.step(val_loss)

    improved = val_loss < best_val
    if improved:
        best_val, best_epoch, stale = val_loss, epoch, 0
        torch.save(model.state_dict(), BEST_PT)
    else:
        stale += 1

    lr_now = optimizer.param_groups[0]["lr"]
    outer.write(
        f"epoch {epoch:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  "
        f"best={best_val:.4f}  stale={stale:2d}  lr={lr_now:.1e}"
        + ("  *" if improved else "")
    )
    outer.set_postfix(
        train=f"{train_loss:.4f}", val=f"{val_loss:.4f}",
        best=f"{best_val:.4f}", stale=stale, lr=f"{lr_now:.1e}",
    )

    if stale >= PATIENCE:
        print(f"Early stop at epoch {epoch}.")
        break


In [ ]:
# --- Test evaluation in original (un-normalised) units --------------------
model.load_state_dict(torch.load(BEST_PT, weights_only=True))
test_mse = evaluate(loaders["test"], mse_metric)

all_pred, all_true = [], []
model.eval()
with torch.no_grad():
    for batch in loaders["test"]:
        batch = batch.to(device, non_blocking=True)
        all_pred.append(model(batch).cpu())
        all_true.append(batch.y.cpu())

pred_norm = torch.cat(all_pred).numpy()
true_norm = torch.cat(all_true).numpy()
pred = pred_norm * y_std.numpy() + y_mean.numpy()
true = true_norm * y_std.numpy() + y_mean.numpy()

names  = ["ux", "uz", "von_mises"]
ss_res = ((true - pred) ** 2).sum(axis=0)
ss_tot = ((true - true.mean(axis=0)) ** 2).sum(axis=0)
r2     = 1 - ss_res / ss_tot
rmse   = np.sqrt(((true - pred) ** 2).mean(axis=0))
mare_vm = float(np.mean(np.abs(pred[:, 2] - true[:, 2]) / (np.abs(true[:, 2]) + 1e-10)))

print("─" * 55)
print(f"MSE (normalised): {test_mse:.4f}")
for i, name in enumerate(names):
    print(f"  {name:12s}: R²={r2[i]:.4f}  RMSE={rmse[i]:.6f}")
print(f"von_mises MARE:   {mare_vm * 100:.2f}%")
print(f"Best epoch:       {best_epoch}")

results = {
    "test_mse_norm":      round(float(test_mse), 6),
    "best_epoch":         best_epoch,
    "mare_von_mises_pct": round(mare_vm * 100, 2),
}
for i, n in enumerate(names):
    results[f"r2_{n}"]   = round(float(r2[i]), 4)
    results[f"rmse_{n}"] = round(float(rmse[i]), 6)

with open(OUT_DIR / "gnn_results.json", "w") as f:
    json.dump(results, f, indent=2)
np.savez(OUT_DIR / "gnn_norm_stats.npz",
         y_mean=y_mean.numpy(), y_std=y_std.numpy())

print()
print("Saved to /kaggle/working/:")
print("  best_gnn.pt           (model weights)")
print("  gnn_results.json      (test metrics)")
print("  gnn_norm_stats.npz    (y_mean, y_std for inference)")
print()
print("Download these from the 'Output' tab in the right sidebar and copy")
print("them into your local ml/ directory.")
